In [2]:
import torch
import torch.nn as nn
import os
import argparse
import pandas as pd
from scipy.io import loadmat
from model.UNet import *
from model.EXNN import *
from model.LinearLMS import *
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from DigiCommPy.modem import Modem,PAMModem

In [3]:
data_tx = loadmat('./exp15_paras.mat')['originPAM']
data_tx = torch.tensor(data_tx.reshape(-1)).float()
data_rx = loadmat('./exp15_CHAN1_2859.mat')['pamRecv']
data_rx = torch.tensor(data_rx.reshape(-1)).float()
data_tx,data_rx.shape,data_rx[0:15].shape

(tensor([ 5.,  5., -1.,  ...,  3.,  5.,  5.]),
 torch.Size([15000]),
 torch.Size([15]))

In [4]:
tapx = 15
tapy = 1
datalen = 15000
trainlen = 10000
stride = 1

In [5]:
model = nn.Linear(15,1)
test_in = torch.randn(15)
model(test_in).shape

torch.Size([1])

In [6]:
# 从start开始，每隔stride取tap个连续数据
def split(data, start, end, stride, tap):
    x = []
    for i in range(start, end - tap + 1, stride):
        x.append(data[i:i+tap])
    return torch.stack(x)

tainRx = split(data_rx, start = 0, end = trainlen, stride = 1, tap = tapx)
trainTx = split(data_tx, start = tapx//2, end = trainlen-tapx//2, stride = 1, tap = tapy)
testRx = split(data_rx, start = trainlen, end = datalen, stride = 1, tap = tapx)
testTx = split(data_tx, start = trainlen+tapx//2, end = datalen-tapx//2, stride = 1, tap = tapy)

In [7]:
trainDataset = torch.utils.data.TensorDataset(tainRx, trainTx)
trainDL = DataLoader(trainDataset, batch_size = 32, shuffle = True)
loss = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
num_epochs = 10
model.train()
for epoch in range(num_epochs):
    for X,y in trainDL:
        optimizer.zero_grad()
        l = loss(model(X),y)
        l.backward()
        optimizer.step()
        
    print(f'epoch:{epoch + 1},loss:{loss(model(tainRx),trainTx):f}')

epoch:1,loss:4.222029
epoch:2,loss:1.160927
epoch:3,loss:0.273199
epoch:4,loss:0.078924
epoch:5,loss:0.051142
epoch:6,loss:0.049280
epoch:7,loss:0.048958
epoch:8,loss:0.048989
epoch:9,loss:0.049130
epoch:10,loss:0.048981


In [8]:
# 比较数据是否正确
def errorNum(rx, tx, threshold=1):
    index = torch.where(torch.abs(rx - tx) > threshold, 1, 0)
    tx = torch.flatten(tx)
    sum = torch.sum(index)
    return (
        sum,
        torch.nonzero(torch.flatten(index)),
        (sum / len(tx)).item(),
    )

In [10]:
model.eval()
testDataset = torch.utils.data.TensorDataset(testRx, testTx)
testDL = DataLoader(testDataset, batch_size = int(1e5), shuffle = False)

for X,y in testDL:
    # print(model(X)[:10])
    # print(y[:10])
    # print(loss(model(X),y))
    # print(errorNum(model(X).clamp(min=-7,max=7), y, threshold=1))
    _, _, ratio = errorNum(model(X).clamp(min=-7,max=7), y, threshold=1)
    pass
ratio

0.001403931062668562

In [51]:

def binary_order(n):
    return len(bin(n)) - 2

def decimalToBinary(decimalData: torch.Tensor, maxNum):
    binary = []
    order = binary_order(maxNum)
    for iOrder in range(order, 0, -1):
        binary.append(
            torch.bitwise_and(torch.bitwise_right_shift(decimalData, iOrder - 1), 1)
        )
    return torch.stack(binary, 1).reshape(-1)
yhat = model(X)
yhat = torch.round((yhat+7)/2).clamp(min=0,max=7).int()
decimalToBinary(yhat, 7)

tensor([0, 1, 1,  ..., 1, 0, 1], dtype=torch.int32)